# ZFIN → ZAPP: the fish lookup tables

**Scope: the easiest case only** — a curator who knows their allele(s) and their
wild-type background. The critical pieces, in order:

1. the **wild-type background table** (the dropdown),
2. the **allele lookup table** (what the search box searches) and its composition,
3. the **gene lookup table** (what a typed gene name resolves against),
4. the **injected-reagent lookup table** (morpholinos / CRISPRs / TALENs) and its
   `reagent_type` enumeration,
5. the **`alteration_type` enumeration** — with the corpus counts that justify it,
6. the **`zygosity` enumeration** — with ZFIN's own observed usage that justifies it,
7. the **mapping codomain** for the two ids stored per fish
   (`genotype_zfin_id`, `fish_zfin_id`) — the registered records they may come from.

The functions here mirror the production service
(`zapp_atlas.api.services.zfin_lookup`) rule for rule, and the last section asserts
the two agree — so this report stays honest without re-implementing the API.

Re-run: `just fetch-zfin` (fresh downloads) then `just qc-report` (renders HTML).

In [1]:
import re
from collections import Counter

import pandas as pd

from zapp_atlas.settings import DEFAULT_ZFIN_DATA_DIR as DATA

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 40)

# ---------------------------------------------------------------------------
# The ZFIN bulk downloads we use. Each file lives at DOWNLOADS_URL + filename;
# `just fetch-zfin` fetches them all into DATA. Files are tab-separated with no
# header row, so next to each filename is the meaning of its leading columns
# (we ignore everything to the right of the ones we name).
# ---------------------------------------------------------------------------
DOWNLOADS_URL = "https://zfin.org/downloads/"

# One row per allele *record* — alleles repeat, see the allele section.
FEATURES_FILE = "features.txt"
FEATURE_COLS = ["allele_id", "so_type", "symbol", "symbol_long", "type_label",
                "mutagen", "treated_with", "construct_id", "construct_name"]

# Which gene an allele damages (only the 'is allele of' rows mean that).
AFFECTED_GENES_FILE = "features-affected-genes.txt"
AFFECTED_GENE_COLS = ["allele_id", "so_type", "symbol",
                      "gene_symbol", "gene_id", "gene_so", "relationship"]

# The wild-type lines: the background dropdown, one row each.
WILDTYPES_FILE = "wildtypes_fish.txt"
WILDTYPE_COLS = ["fish_id", "name", "abbreviation", "genotype_id"]

# Every marker ZFIN tracks (genes, transcripts, constructs, ...), typed.
MARKERS_FILE = "genetic_markers.txt"
MARKER_COLS = ["gene_id", "symbol", "name", "marker_type", "so_type"]

# Registered allele combinations; the identity string carries the
# [fish,mother,father] zygosity triple per allele.
GENOTYPES_FILE = "genotype_features.txt"
GENOTYPE_COLS = ["genotype_id", "display_name", "identity", "allele_id"]

# Genotype -> its wild-type background; backgrounds are themselves ZDB-GENO records.
BACKGROUNDS_FILE = "genotype_backgrounds.txt"
BACKGROUND_COLS = ["genotype_id", "genotype_name", "background_id", "background_name"]

# One row per (fish, component): every registered fish, its genotype, and any
# transient reagents (morpholino / CRISPR / TALEN) it carries.
FISH_FILE = "fish_components_fish.txt"
FISH_COLS = ["fish_id", "fish_name", "gene_id", "gene_symbol",
             "component_id", "component_symbol", "construct_id", "construct_name",
             "background_id", "background_name", "genotype_id"]

# The registered transient reagents, one file per kind. One row per
# (reagent, targeted gene) — a few hundred reagents hit several genes, so
# downstream code aggregates per reagent id. TALEN rows carry a second
# target-arm sequence in a column we don't read.
MORPHOLINOS_FILE = "Morpholinos.txt"
CRISPR_FILE = "CRISPR.txt"
TALEN_FILE = "TALEN.txt"
REAGENT_COLS = ["gene_id", "gene_so", "gene_symbol",
                "reagent_id", "reagent_so", "reagent_symbol"]

# Reagent component ids in fish_components carry these prefixes.
STR_PREFIXES = ("ZDB-MRPHLNO", "ZDB-CRISPR", "ZDB-TALEN")


def read_tsv(filename, columns):
    """Read one ZFIN download, naming only the leading columns we use."""
    return pd.read_csv(DATA / filename, sep="\t", header=None,
                       names=columns, usecols=range(len(columns)),
                       dtype=str, quoting=3)


# alteration_type values are defined in the schema YAML, each carrying its
# Sequence Ontology term as a `meaning:`. Inverted here into SO -> enum value;
# the data-driven justification for the value set is in the Enumerations section.
from linkml_runtime import SchemaView
from zapp_atlas.schema.constraints import SCHEMA_PATH

_enum = SchemaView(str(SCHEMA_PATH)).get_enum("SequenceAlterationTypeEnum")
SO_TO_ALTERATION = {pv.meaning: value for value, pv in _enum.permissible_values.items()
                    if pv.meaning}

/Users/ao33/Desktop/ZAPP_DATA_MODEL_FISH_UPDATE/zapp-atlas/server/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Lookup table 1 — wild-type backgrounds

**File:** `wildtypes_fish.txt`. This *is* the background dropdown: pick a name, and
the strain's genotype id (and, for a pure wild-type fish, its fish id) attach. The
form adds one non-ZFIN option, **"unknown"**, stored as an omitted field — no record
is ever guessed. Stored ids get a `ZFIN:` prefix (CURIE).

In [2]:
def load_wildtypes():
    """The background dropdown, straight from the file."""
    return read_tsv(WILDTYPES_FILE, WILDTYPE_COLS).sort_values("name",
        key=lambda names: names.str.lower()).reset_index(drop=True)


wildtypes = load_wildtypes()
wildtypes

,fish_id,name,abbreviation,genotype_id
0,ZDB-FISH-150901-27842,AB,AB,ZDB-GENO-960809-7
1,ZDB-FISH-150901-19012,AB/C32,AB/C32,ZDB-GENO-070425-3
2,ZDB-FISH-150901-18519,AB/EKW,AB/EKW,ZDB-GENO-091223-1
3,ZDB-FISH-150901-29235,AB/TL,AB/TL,ZDB-GENO-031202-1
4,ZDB-FISH-150901-29084,AB/TU,AB/TU,ZDB-GENO-010924-10
5,ZDB-FISH-181106-1,ABO,ABO,ZDB-GENO-181106-1
6,ZDB-FISH-150901-28222,C32,C32,ZDB-GENO-030501-1
7,ZDB-FISH-180717-1,Cooch Behar,CB,ZDB-GENO-180717-1
8,ZDB-FISH-150901-28033,DAR,DAR,ZDB-GENO-960809-13
9,ZDB-FISH-150901-29731,EKW,EKW,ZDB-GENO-990520-2


## Lookup table 2 — alleles

**Files:** `features.txt` joined with `features-affected-genes.txt`.

Two file quirks the functions handle, both verified against the data:

- `features.txt` **repeats ~1.7k alleles** — identically, or once per construct
  (a co-injected line is ONE insertion carrying several constructs) → aggregate
  per allele id.
- In `features-affected-genes.txt`, only rows whose relationship is
  **`is allele of`** link an allele to the gene it damages (the rest is curation
  bookkeeping) → keep only those; one gene per allele (just 2 of 59k have more).

In [3]:
SO_TRANSGENIC_INSERTION = "SO:0001218"


def load_affected_genes():
    """allele_id -> the gene it damages, one row per allele."""
    genes = read_tsv(AFFECTED_GENES_FILE, AFFECTED_GENE_COLS)
    genes = genes[genes["relationship"] == "is allele of"]
    return genes.drop_duplicates("allele_id")[["allele_id", "gene_symbol", "gene_id"]]


def load_alleles():
    """One row per allele: the table the search box searches."""
    rows = read_tsv(FEATURES_FILE, FEATURE_COLS)
    rows = rows[rows["allele_id"].str.startswith("ZDB-ALT-", na=False)]

    constructs = (rows[["allele_id", "construct_id", "construct_name"]].dropna()
                  .drop_duplicates(["allele_id", "construct_id"])
                  .groupby("allele_id")
                  .agg(construct_name=("construct_name", " + ".join),
                       n_constructs=("construct_id", "size")))

    alleles = rows.drop_duplicates("allele_id")
    alleles = alleles[["allele_id", "symbol", "so_type", "type_label", "mutagen"]]
    alleles = alleles.merge(constructs, on="allele_id", how="left")
    alleles = alleles.merge(load_affected_genes(), on="allele_id", how="left")

    alleles["mutagen"] = alleles["mutagen"].replace("not specified", pd.NA)
    alleles["n_constructs"] = alleles["n_constructs"].fillna(0).astype(int)
    alleles["alteration_type"] = alleles["so_type"].map(SO_TO_ALTERATION)
    alleles["is_transgenic"] = ((alleles["so_type"] == SO_TRANSGENIC_INSERTION)
                                | (alleles["n_constructs"] > 0))
    return alleles


alleles = load_alleles()
alleles[alleles["symbol"].isin(["fh111", "w200Tg", "gz13Tg"])]

,allele_id,symbol,so_type,type_label,mutagen,construct_name,n_constructs,gene_symbol,gene_id,alteration_type,is_transgenic
585,ZDB-ALT-160602-14,fh111,SO:1000008,Allele with one point mutation,NaN,NaN,0,snapc1b,ZDB-GENE-040426-716,point_mutation,False
53144,ZDB-ALT-090312-1,gz13Tg,SO:0001218,Transgenic Insertion,DNA,Tg2(krt4:GFP) + Tg2(mylpfa:RFP),2,NaN,NaN,transgenic_insertion,True
75013,ZDB-ALT-130130-3,w200Tg,SO:0001218,Transgenic Insertion,DNA,Tg(mpeg1:YFP),1,NaN,NaN,transgenic_insertion,True


### Composition of the allele table

The same numbers, in the data model's terms: which card a search hit becomes
(`TransgenicAllele` vs `MutantAllele`) and how complete each card arrives.

In [4]:
transgenic = alleles["is_transgenic"]
placeholder = alleles["symbol"].str.endswith(("_unspecified", "_unrecovered"))
has_gene = alleles["gene_symbol"].notna()

pd.DataFrame([
    ["alleles, total", len(alleles)],
    ["→ TransgenicAllele cards", transgenic.sum()],
    ["    with construct name + id", (transgenic & alleles["construct_name"].notna()).sum()],
    ["    co-injected (2 constructs)", (alleles["n_constructs"] > 1).sum()],
    ["→ MutantAllele cards", (~transgenic).sum()],
    ["    with the damaged gene attached", (~transgenic & has_gene).sum()],
    ["    gene unknown (shown honestly)", (~transgenic & ~has_gene).sum()],
    ["    gene-level placeholders (_unspecified / _unrecovered)", placeholder.sum()],
], columns=["what", "count"])

,what,count
0,"alleles, total",81170
1,→ TransgenicAllele cards,28583
2,with construct name + id,28293
3,co-injected (2 constructs),80
4,→ MutantAllele cards,52587
5,with the damaged gene attached,52145
6,gene unknown (shown honestly),442
7,gene-level placeholders (_unspecified / _unrecovered),176


## Lookup table 3 — genes

**File:** `genetic_markers.txt` — every marker ZFIN tracks, one row each, with a
`marker_type`.

**Namespace:** we use **ZFIN official gene symbols** for humans (ZFIN is the
zebrafish nomenclature authority — it *assigns* the symbols) and **ZDB-GENE ids**
(stored as `ZFIN:` CURIEs) as the identifier. Symbols are current as of the
download; ZFIN publishes historical symbols in `aliases.txt`, should renamed
genes ever need handling (not fetched — nothing consumes it).

**Which types count as a "gene" — and the data behind it:** real alleles damage
more than protein-coding genes. Of the gene ids referenced by
`features-affected-genes.txt`, 157 are miRNA genes, lincRNA genes, pseudogenes, or
regulatory regions. The supported universe is therefore the **gene-like set** below
— referential integrity 100%, versus 99.23% for `GENE` alone.

In [5]:
GENE_LIKE = {
    "GENE": "protein-coding gene",
    "GENEP": "pseudogene",
    "MIRNAG": "microRNA gene",
    "LINCRNAG": "lincRNA gene",
    "LNCRNAG": "lncRNA gene",
    "NCRNAG": "ncRNA gene",
    "SNORNAG": "snoRNA gene",
    "ENHANCER": "enhancer",
    "NCCR": "non-coding control region",
}


def load_genes():
    """The searchable gene universe: every gene-like ZFIN marker."""
    markers = read_tsv(MARKERS_FILE, MARKER_COLS)
    genes = markers[markers["marker_type"].isin(GENE_LIKE)]
    return genes[["gene_id", "symbol", "name", "marker_type"]]


genes = load_genes()

breakdown = genes["marker_type"].value_counts().rename_axis("marker_type").reset_index(name="genes")
breakdown["meaning"] = breakdown["marker_type"].map(GENE_LIKE)
print(f"{len(genes):,} searchable genes")
display(breakdown)

referenced = load_affected_genes()["gene_id"]
print(f"referential integrity: {referenced.isin(genes['gene_id']).mean():.2%} "
      "of gene ids referenced by alleles exist in this table")

38,347 searchable genes


,marker_type,genes,meaning
0,GENE,36049,protein-coding gene
1,LINCRNAG,915,lincRNA gene
2,MIRNAG,426,microRNA gene
3,GENEP,418,pseudogene
4,NCCR,209,non-coding control region
5,ENHANCER,186,enhancer
6,LNCRNAG,60,lncRNA gene
7,NCRNAG,51,ncRNA gene
8,SNORNAG,33,snoRNA gene


referential integrity: 100.00% of gene ids referenced by alleles exist in this table


## Lookup table 4 — injected reagents (the transient layer)

A fish can also carry transient reagents — morpholinos, CRISPRs and TALENs
injected into the embryos. They are not heritable, so they sit on the **fish**,
not the genotype: there is no zygosity to record, and dose / injection stage
belong to the experiment's exposure record. The form's reagent search box draws
on the full registries below, not just reagents already seen in a registered
fish — a curator may be the first to describe an injection. Every row carries
the target gene, so picking a reagent auto-fills its gene exactly the way
picking an allele does.

`reagent_type` is a schema enum with one value per registry. ZFIN's own SO
typing (shown below) supports a `meaning:` only for morpholinos —
`SO:0000034` is *morpholino_oligo*, a reagent kind. CRISPR and TALEN rows are
typed with *binding-site* terms (`DNA_binding_site`, `nuclease_binding_site`),
which describe a place in the genome rather than a reagent, so those two enum
values stay label-only instead of borrowing a wrong term.


In [6]:
def load_reagents():
    """The registered transient reagents: three files sharing one layout."""
    frames = [
        read_tsv(filename, REAGENT_COLS).assign(kind=kind)
        for filename, kind in [(MORPHOLINOS_FILE, "morpholino"),
                               (CRISPR_FILE, "crispr"),
                               (TALEN_FILE, "talen")]
    ]
    table = pd.concat(frames, ignore_index=True)
    # Mirror the production parser: drop the rare malformed row w/o a ZDB id.
    return table[table["reagent_id"].str.startswith("ZDB-", na=False)]


reagents = load_reagents()

# In use = appears as a component of at least one registered fish.
components = read_tsv(FISH_FILE, FISH_COLS)["component_id"].fillna("")
used = set(components[components.str.startswith(STR_PREFIXES)])

ids = reagents.drop_duplicates("reagent_id")[["reagent_id", "kind"]].copy()
targets = reagents.groupby("reagent_id")["gene_id"].nunique()
ids["in_use"] = ids["reagent_id"].isin(used)
ids["multi_target"] = ids["reagent_id"].map(targets) > 1

print(f"registered reagents: {len(ids):,}   in use in a fish: {int(ids.in_use.sum()):,}")
print(f"target-gene integrity vs the gene table: "
      f"{reagents['gene_id'].isin(genes['gene_id']).mean():.2%}")
print("\nZFIN's own SO typing per kind (the meaning: evidence):")
print(reagents.groupby("kind")["reagent_so"].value_counts().to_string())

ids.groupby("kind").agg(registered=("reagent_id", "size"),
                        in_use=("in_use", "sum"),
                        multi_target=("multi_target", "sum")
                        ).reindex(["morpholino", "crispr", "talen"])


registered reagents: 31,545   in use in a fish: 11,001
target-gene integrity vs the gene table: 99.94%

ZFIN's own SO typing per kind (the meaning: evidence):
kind        reagent_so
crispr      SO:0001429    18815
morpholino  SO:0000034    12545
talen       SO:0000059     1098


,registered,in_use,multi_target
kind,,,
morpholino,12030,8874,126
crispr,18530,2088,107
talen,985,39,9


## Enumeration 1 — `alteration_type`, and why exactly these values

The values live in **one place: the schema YAML** (`SequenceAlterationTypeEnum`),
each carrying its Sequence Ontology term as a `meaning:` (the map was built in the
setup cell). The value set is not taste — the two tables below are the evidence:

- **Nine values carry 99.95% of ZFIN's corpus** — from `point_mutation` (38.5k
  alleles) down to `translocation` (26). Three of them — `deficiency`,
  `translocation`, `multiple_variants` — were **promoted from the label-only list
  on 2026-09-04**, the day this report first quantified them: the promotion
  process working as designed.
- **Four values have zero ZFIN usage at the allele level** (`substitution`,
  `complex_substitution`, `inversion`, `duplication`) — the zeros are shown, not
  hidden. They exist for the *new-allele* form: standard SO alteration classes a
  biologist may need when describing an allele ZFIN hasn't typed yet.
- **The label-only remainder is now 39 alleles**: "Allele with one mnv" (37 —
  ZFIN gives those rows no SO term to map from) and "Unspecified" (2). Those cards
  stay fully valid — ZFIN's label shows, the id points at the complete record. If
  the mnv row grows, promoting it means mapping by label (or ZFIN adopting SO's
  MNV term), a deliberate extra step.

In [7]:
from zapp_atlas.schema.pydantic_crud import SequenceAlterationTypeEnum

counts = alleles["alteration_type"].value_counts()
support = pd.DataFrame({"alteration_type": [v.value for v in SequenceAlterationTypeEnum]})
support["SO term"] = support["alteration_type"].map(
    {value: so for so, value in SO_TO_ALTERATION.items()})
support["alleles in ZFIN"] = support["alteration_type"].map(counts).fillna(0).astype(int)
print(f"{alleles['alteration_type'].notna().mean():.2%} of alleles map to an enum value:")
display(support.sort_values("alleles in ZFIN", ascending=False))

print("label-only remainder, and what those records actually are:")
(alleles[alleles["alteration_type"].isna()]
 .groupby(["so_type", "type_label"], dropna=False).size()
 .reset_index(name="alleles").sort_values("alleles", ascending=False))

99.95% of alleles map to an enum value:


,alteration_type,SO term,alleles in ZFIN
0,point_mutation,SO:1000008,38529
7,transgenic_insertion,SO:0001218,28583
2,deletion,SO:0000159,6419
12,sequence_alteration,SO:0001059,3760
4,indel,SO:1000032,2334
3,insertion,SO:0000667,992
11,multiple_variants,SO:0001023,377
9,deficiency,SO:1000029,111
10,translocation,SO:0000199,26
1,substitution,SO:1000002,0


label-only remainder, and what those records actually are:


,so_type,type_label,alleles
0,NaN,Allele with one mnv,37
1,NaN,Unspecified,2


## Enumeration 2 — `zygosity`, and why exactly these values

Also defined in the schema YAML (`ZygosityEnum`). Zygosity is always the
**curator's assertion** — nothing looks it up. The value set follows ZFIN's own
vocabulary: its genotype identity strings give every allele a
`[fish, mother, father]` code triple, and the observed usage below is the evidence
for each of our four values — and for the defaults the form picks.

In [8]:
from zapp_atlas.schema.pydantic_crud import ZygosityEnum

ZFIN_CODE_TO_OURS = {"2": "homozygous", "1": "heterozygous",
                     "U": "unknown", "W": "wild_type"}

print("our values:", [value.value for value in ZygosityEnum])


def observed_zygosity_codes():
    """Count the [fish,mother,father] codes across all registered genotypes."""
    counts = {"fish": Counter(), "mother": Counter(), "father": Counter()}
    for identity in read_tsv(GENOTYPES_FILE, GENOTYPE_COLS)["identity"].dropna():
        for triple in re.findall(r"\[([^\]]*)\]", identity):
            codes = [part.strip() for part in triple.split(",")]
            if len(codes) == 3:
                for who, code in zip(counts, codes):
                    counts[who][code] += 1
    table = pd.DataFrame(counts).fillna(0).astype(int)
    table.insert(0, "our value", table.index.to_series().map(ZFIN_CODE_TO_OURS)
                 .fillna("— not modeled —"))
    return table


observed_zygosity_codes()

our values: ['homozygous', 'heterozygous', 'unknown', 'wild_type']


,our value,fish,mother,father
U,unknown,44541,66241,66692
2,homozygous,36864,2557,1679
1,heterozygous,14256,26707,26985
C,— not modeled —,23,3,1
W,wild_type,0,176,327


Reading that table, value by value:

- **`homozygous` / `heterozygous`** — the workhorses for the fish itself (37k / 14k).
- **`unknown` is the honest default**: it dominates even for the fish (44k), and
  overwhelmingly for parents (~66k each) — which is why the form's parent fields
  *default* to unknown rather than demanding an answer.
- **`wild_type` earned its place**: W appears only in parental slots (176 mothers,
  327 fathers — e.g. a het fish from a het × wild-type cross) and **never** for the
  fish itself: a fish wild-type for an allele simply doesn't list the allele. Without
  W those crosses were inexpressible; that is why the enum grew it.
- **C (complex) is deliberately not modeled**: ~27 occurrences in ~184k slots. If
  that row grows, revisit.

## The two mapped identifiers — what a fish is allowed to map to

ZAPP's `Fish` is flat: the curator enters the allele(s) and the background, and
the record carries two identifier slots — `genotype_zfin_id` (ZDB-GENO-…) and
`fish_zfin_id` (ZDB-FISH-…) — that are **mapped, never typed**. In ZFIN's system
a *genotype* is a registered allele combination (each allele with its
[fish, mother, father] zygosity triple, on a background), and a *fish* is a
genotype plus any transient reagents (morpholino / CRISPR / TALEN injections).
ZAPP models the injected reagents too, so both halves of the fish registry are
in reach: a reagent-free fish is named by its genotype alone, and a
reagent-carrying fish by its genotype plus its reagent set.


In [9]:
def load_genotypes():
    """One row per (genotype, allele): the registered allele combinations."""
    return read_tsv(GENOTYPES_FILE, GENOTYPE_COLS)


def load_backgrounds():
    """Genotype -> its registered wild-type background (itself a ZDB-GENO)."""
    return read_tsv(BACKGROUNDS_FILE, BACKGROUND_COLS)


genotypes = load_genotypes()
# Distinct alleles, not rows: a handful of genotypes repeat an allele row.
alleles_per_genotype = genotypes.groupby("genotype_id")["allele_id"].nunique()

print(f"registered genotypes (allele-carrying): {alleles_per_genotype.size:,}")
print(f"with a registered background:           {load_backgrounds()['genotype_id'].nunique():,}")
print(f"wild-type lines (their own GENO ids):   {len(load_wildtypes()):,}")

combination_size = alleles_per_genotype.clip(upper=4).value_counts().sort_index()
combination_size.index = ["1", "2", "3", "4+"]
combination_size.rename_axis("alleles in the combination").to_frame("genotypes")


registered genotypes (allele-carrying): 38,141
with a registered background:           12,001
wild-type lines (their own GENO ids):   32


,genotypes
alleles in the combination,
1,25751
2,9439
3,2414
4+,537


In [10]:
def load_fish():
    """One row per (fish, component): every registered fish and what it carries."""
    return read_tsv(FISH_FILE, FISH_COLS)


fish = load_fish()
is_str = fish["component_id"].fillna("").str.startswith(STR_PREFIXES)
has_reagent = is_str.groupby(fish["fish_id"]).any()

print(f"registered fish records:       {has_reagent.size:,}")
print(f"  carrying transient reagents: {int(has_reagent.sum()):,}  (named by genotype + reagent set)")
print(f"  reagent-free:                {int((~has_reagent).sum()):,}  (named by genotype alone)")

# The fact the flat model leans on: a genotype has AT MOST ONE reagent-free fish.
clean = fish[~fish["fish_id"].map(has_reagent)]
fish_per_genotype = clean.groupby("genotype_id")["fish_id"].nunique()
assert (fish_per_genotype <= 1).all(), "one genotype, one reagent-free fish no longer holds"
covered = alleles_per_genotype.index.isin(fish_per_genotype.index).sum()
print(f"\ngenotypes with more than one reagent-free fish: {int((fish_per_genotype > 1).sum())}")
print(f"allele-carrying genotypes with a reagent-free fish: "
      f"{covered:,} of {alleles_per_genotype.size:,}")

# A reagent-carrying fish is named by (genotype, reagent set).
carrying = fish[fish["fish_id"].map(has_reagent)]


def identity(group):
    genotype = group["genotype_id"].dropna().unique()
    reagent_set = frozenset(
        group.loc[group["component_id"].fillna("").str.startswith(STR_PREFIXES), "component_id"]
    )
    return (genotype[0] if len(genotype) else None, reagent_set)


identities = carrying.groupby("fish_id").apply(identity, include_groups=False)
shared = identities.value_counts()
duplicated = shared[shared > 1]
print(f"\nreagent-carrying fish: {identities.size:,}")
print(f"(genotype, reagent set) keys naming more than one fish: {len(duplicated)}")
for key, count in duplicated.items():
    names = fish.loc[fish["fish_id"].isin(identities[identities == key].index), "fish_name"]
    print(f"  {count} fish ids for {names.iloc[0]!r} — duplicate records on ZFIN's side")
assert len(duplicated) <= 2, "reagent-fish identity degraded beyond the known ZFIN duplicates"


registered fish records:       56,525
  carrying transient reagents: 18,821  (named by genotype + reagent set)
  reagent-free:                37,704  (named by genotype alone)

genotypes with more than one reagent-free fish: 0
allele-carrying genotypes with a reagent-free fish: 37,704 of 38,141



reagent-carrying fish: 18,821
(genotype, reagent set) keys naming more than one fish: 2
  2 fish ids for 'AB + MO1-gata1a' — duplicate records on ZFIN's side
  2 fish ids for 'AB + MO1-etsrp' — duplicate records on ZFIN's side


Reading those numbers:

- **`genotype_zfin_id`** may hold any registered combination's id, or a
  wild-type line's own genotype id. **`background_zfin_id`** draws on the same
  registry — a background *is* a genotype record, which is why the two slots
  share the ZDB-GENO pattern.
- **`fish_zfin_id`**: for a reagent-free fish the genotype match *determines*
  the fish id — the mapping is 1:1 per genotype (the assert above), so two
  scalar slots are enough and there is no candidate list to store. A
  reagent-carrying fish is named by genotype + reagent set instead — unique in
  all but two cases, which are literal duplicate records on ZFIN's side (the
  resolver picks deterministically, lowest id, and logs it).
- The few hundred genotypes with no reagent-free fish record get
  `genotype_zfin_id` without `fish_zfin_id`. Both slots are optional — absence
  is honest.
- Nothing here computes the ids for a curator's entry: matching an entered
  composition against these registries is the resolver, the next feature. This
  section pins down its codomain — what it is allowed to answer with.


## Cross-check: these tables agree with the production API

The service (`zfin_lookup.get_index`) builds its index by the same rules. If a
refactor ever changes one side, this cell fails.

In [11]:
from zapp_atlas.api.services.zfin_lookup import get_index

index = get_index(DATA)
assert index.allele_count == len(alleles)
assert len(index.wildtypes) == len(wildtypes)

fh111 = index.search_alleles("fh111", 1)[1][0]
row = alleles.set_index("symbol").loc["fh111"]
assert fh111.allele_id == "ZFIN:" + row["allele_id"]        # stored form is a CURIE
assert fh111.affected_genes[0].gene_symbol == row["gene_symbol"]
assert fh111.alteration_type.value == row["alteration_type"]

print("notebook tables agree with the production index ✓")

# The reagent table agrees with the production index too.
assert index.reagent_count == reagents["reagent_id"].nunique()
_, [mo1] = index.search_reagents("MO1-gata1a", 1)
mo1_row = reagents[reagents["reagent_symbol"] == "MO1-gata1a"].iloc[0]
assert mo1.reagent_id == "ZFIN:" + mo1_row["reagent_id"]
assert mo1.reagent_type.value == "morpholino"
print("reagent cross-check passed:", f"{index.reagent_count:,} reagents")


notebook tables agree with the production index ✓
reagent cross-check passed: 31,545 reagents


## Decisions on record (2026-09 snapshot)

- **Aggregate `features.txt` per allele id** — 1,755 repeated ids; 80 co-injected
  lines carry two constructs under one allele (e.g. `gz13Tg`).
- **Gene link = `is allele of` rows only, one gene per allele** — 2 of 59k alleles
  touch more than one gene; scalar slots are fine.
- **Gene universe = the gene-like marker types** (protein-coding + ncRNA genes,
  pseudogenes, regulatory regions) — 157 referenced ids are non-`GENE`; integrity
  100% vs 99.23% for `GENE` alone. Symbols are ZFIN's official assignments; stored
  ids are `ZFIN:ZDB-GENE-…` CURIEs.
- **`alteration_type` values live in the schema YAML as SO meanings** — nine values
  carry 99.95% of the corpus (deficiency / translocation / multiple_variants
  promoted 2026-09-04 off this report's evidence); four zero-usage values are kept
  for describing new alleles; only mnv (37) and Unspecified (2) stay label-only.
- **Enum vs data, the placement rule**: a small closed classification a curator
  picks from → schema enum with ontology meanings (alteration type, zygosity,
  progeny selection). A large open vocabulary with an external authority → CURIE +
  symbol slots in the schema, vocabulary served as lookup data (alleles, genes,
  backgrounds — and chemicals, on the exposure side).
- **Zygosity is a curator assertion** — parents default `unknown` (U dominates
  ZFIN's own records); `wild_type` = ZFIN's W, parental-only in practice; the rare
  C (complex, ~27×) is deliberately unmodeled until it matters.
- **Fish is flat — no Genotype object** (2026-09-04): `genotype_name` always
  duplicated the fish name and the background recursion never went past one
  level, so Fish carries the allele lists, `background_name`/`background_zfin_id`,
  and the two mapped ids directly.
- **The two ZFIN ids are mapped, never entered** — their codomain is the
  registered records: `genotype_features` for `genotype_zfin_id`; the
  reagent-free rows of `fish_components_fish` for `fish_zfin_id` (1:1 per
  genotype, asserted in this report; reagent-carrying fish resolve by
  genotype + reagent set, with two known ZFIN-side duplicates); `wildtypes_fish`
  for pure wild-type subjects.
- **Transient reagents live on the fish** (2026-09-08): a third list beside the
  allele lists — no zygosity (nothing heritable to be zygous about), no dose
  (that is the exposure record's business). The dropdown draws on the full
  Morpholinos/CRISPR/TALEN registries, aggregated per reagent id (a few hundred
  hit several genes). `reagent_type`'s three values mirror the registries; only
  morpholino carries an SO meaning — ZFIN types CRISPR/TALEN by binding-site
  terms, so those stay label-only.
- **Matching is exact-identity, not fuzzy** (2026-09-08): the curator's entry
  IS the identity — `unknown` is a value (ZFIN's own U), never a wildcard — so
  resolution is a lookup with 0 or 1 answer. No ranked candidates, no
  disambiguation step. Not found = the mapped ids stay absent (never a
  placeholder), which doubles as the "needs minting" worklist.
- **Fish has no name slot** (2026-09-08): display strings are derived from the
  description. The human handle is the tank entry's `nickname` — what the
  group calls the line, unique within the group, the key its picker shows.
- **"Unknown" background = omitted field** — absence is unambiguous; no guessed
  records.

---
*Regenerate: `just fetch-zfin && just qc-report`.*